# Reproducible results: 780,000 building inspections

This notebook computes every number in the article
**"What Do 780,000 Building Inspections Tell Us About New York City's Housing?"**
Each result below is linked from the matching figure in the post: click a number
in the article and you land on the cell that produces it.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/michaelzoorob/dob-complaints-analysis/blob/main/notebooks/overview_results.ipynb)

**How to read this.** Every regression is refit here on the committed property
panel, and every descriptive rate is recomputed from it, so the coefficient,
95% confidence interval, and sample size you see are produced by the cell, not
copied in. A handful of descriptive counts come from the full 8 GB complaint
database (the disposition mix, penalty totals, text-field coverage, monthly
volume); that file is too large for the repository, so those cells display the
committed intermediate table written by the named script and point to the
database snapshot in the article's appendix for full re-derivation. Two owner
models (fewer DOB violations, and the same gap without commercial controls) rest
on the deduplicated DOB-violation count built from that database and are handled
the same way.

**To run it yourself.** Open in Colab (badge above) or clone the repo and run top
to bottom. The pinned environment is in `notebooks/requirements.txt`; only
`pandas`, `numpy`, and `pyfixest` are needed, and the panel file ships in the
repository.

In [1]:
# === setup: versions, committed panel, analysis frame, helpers ===
import sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, pyfixest as pf
import scipy
warnings.filterwarnings("ignore")
print("python", sys.version.split()[0], "| pyfixest", pf.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__, "| scipy", scipy.__version__)

# locate repo root whether run from repo root, notebooks/, or Colab
CAND = [Path.cwd(), Path.cwd().parent] + list(Path.cwd().parents)
ROOT = next((p for p in CAND if (p / "data" / "analysis").exists()), Path.cwd())
DATA = ROOT / "data" / "analysis"
RM = DATA / "risk_models"
DB = ROOT / "data" / "dob_complaints.db"      # optional 8 GB snapshot (article appendix)
HAVE_DB = DB.exists()
YEARS = 6.25                                  # ECB window: Jan 2020 - Mar 2026
PANEL = DATA / "property_risk_panel_v2.csv.gz"
DTYPE = {"bct2020": str, "size_bin": str, "borocode": str, "bbl_key": str}

def build_frame(path):
    """Estimation frame from the committed panel, no database. Mirrors load_frame()
    in risk_factor_models.py / violation_rate_models.py, minus the DOB-violation
    union join (the only step needing the 8 GB database)."""
    df = pd.read_csv(path, dtype=DTYPE, low_memory=False)
    for t in ["llc", "corp_other", "trust_estate", "nycha", "govt"]:
        df[t] = (df["owner_type"] == t).astype(int)
    df = df[df["owner_type"] != "missing"].copy()
    for b in ["owner_occ_star", "is_coop", "is_condo"]:
        df[b] = df[b].astype(int)
    yb = df["yearbuilt"]
    df["era_pre1940"] = yb.between(1800, 1939).astype(int)
    df["era_4079"] = yb.between(1940, 1979).astype(int)
    df["era_8099"] = yb.between(1980, 1999).astype(int)
    df["era_unknown"] = (~yb.between(1800, 2026)).astype(int)
    df["multi_bldg"] = (df["numbldgs"] >= 2).astype(int)
    df["log2_area_per_unit"] = np.log2(df["area_per_unit"])
    # commercial-exposure controls: commercial units binned symmetrically to
    # residential size (fixed effects) + S/K/O building-class dummy + log floor area
    ut = pd.to_numeric(df["unitstotal"], errors="coerce")
    ur = pd.to_numeric(df["unitsres"], errors="coerce")
    df["unitscom"] = np.maximum(ut - ur, 0).fillna(0.0)
    cb = [-1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 25, 50, 100, 250, 100000]
    cl = ["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "10",
          "11-15", "16-25", "26-50", "51-100", "101-250", "251+"]
    df["comm_bin"] = pd.cut(df["unitscom"], bins=cb, labels=cl).astype(str)
    ba = pd.to_numeric(df["bldgarea"], errors="coerce")
    df["log_bldgarea"] = np.log(ba.where(ba > 0))
    df["com_class"] = df["bldgclass"].astype(str).str[0].isin(["S", "K", "O"]).astype(int)
    for g in ["nyc_other", "outside_nyc", "unknown"]:
        df[f"geo_{g}"] = (df["owner_geo"] == g).astype(int)
    df["multi_prop_owner"] = df["multi_prop_owner"].astype(int)
    with np.errstate(invalid="ignore", divide="ignore"):
        df["violrate100"] = np.where(df["n_substantive"] > 0,
                                     df["n_viol_disp"] / df["n_substantive"] * 100.0, np.nan)
        for g in ["conv", "constr", "elev", "boiler"]:
            df[f"sh_{g}"] = np.where(df["n_complaints"] > 0,
                                     df[f"n_{g}"] / df["n_complaints"], 0.0)
    need = ["log2_area_per_unit", "value_rank", "size_bin", "bct2020"]
    keep = df[need].notna().all(axis=1) & np.isfinite(df["log2_area_per_unit"])
    return df[keep].copy()

panel = pd.read_csv(PANEL, dtype=DTYPE, low_memory=False)   # full 766,939-lot panel
frame = build_frame(PANEL)                                  # estimation sample

BUILDING = ["llc", "corp_other", "trust_estate", "nycha", "govt", "owner_occ_star",
            "is_coop", "is_condo", "era_pre1940", "era_4079", "era_8099", "era_unknown",
            "mzone", "multi_bldg", "com_class", "log_bldgarea", "log2_area_per_unit",
            "value_rank", "any_prior_viol"]
OWNER = ["geo_nyc_other", "geo_outside_nyc", "geo_unknown", "multi_prop_owner"]
CAT = ["sh_conv", "sh_constr", "sh_elev", "sh_boiler"]
FE = "size_bin + comm_bin + bct2020"
VCOV = {"CRV1": "bct2020"}
X, XO, XC = " + ".join(BUILDING), " + ".join(BUILDING + OWNER), " + ".join(BUILDING + CAT)

def irr(m, term):
    """Poisson term -> (percent change, CI low, CI high)."""
    b = m.coef()[term]; lo, hi = m.confint().loc[term].values
    return (np.exp(b) - 1) * 100, (np.exp(lo) - 1) * 100, (np.exp(hi) - 1) * 100

def pp(m, term):
    """LPM term -> (points, CI low, CI high)."""
    b = m.coef()[term]; lo, hi = m.confint().loc[term].values
    return b, lo, hi

def committed(csv):
    return pd.read_csv(RM / csv)

print(f"panel: {len(panel):,} lots | estimation frame: {len(frame):,} | database present: {HAVE_DB}")

python 3.14.2 | pyfixest 0.60.0 | pandas 3.0.0 | numpy 2.3.5 | scipy 1.17.0


panel: 766,939 lots | estimation frame: 766,429 | database present: True


## Panel universe and coverage

Lot universe, residential units, deed-address owner-geography coverage, and the count and share of scraped complaints that match a residential lot -- the 83% figure (`panel_headline_counts.py`, `post0_descriptive_stats.py`).

In [2]:
phc = committed("panel_headline_counts.csv").set_index("metric")["value"]
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
print(f"RESULT residential lots: {len(panel):,}")
print(f"RESULT residential units: {panel['unitsres'].sum()/1e6:.2f} million")
print(f"RESULT owner-geography coverage: {phc['owner_geo_coverage_share']*100:.0f}%")
matched, scraped = d["matched_residential_pages"], d["scraped_pages"]
print(f"RESULT complaints matched to a residential lot: {matched:,.0f}"
      f"  ({matched/scraped*100:.0f}% of {scraped:,.0f} scraped)")
# the two producing scripts must agree on the matched count, and the committed
# share must equal the ratio recomputed here
assert int(matched) == int(phc["matched_complaints_2020_may2026"])
assert abs(d["matched_residential_share"] - matched/scraped) < 1e-6

RESULT residential lots: 766,939
RESULT residential units: 3.74 million
RESULT owner-geography coverage: 70%
RESULT complaints matched to a residential lot: 648,363  (83% of 783,289 scraped)


## Complaint volume since 2020

Monthly and daily complaint flow and the count of distinct buildings, Jan 2020 - May 2026. Computed by `post0_descriptive_stats.py` from the complaint database; displayed here from the committed table.

In [3]:
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
print(f"RESULT complaints per month (avg): {d['monthly_average']:,.0f}")
print(f"RESULT complaints per day (avg): {d['daily_average']:,.0f}")
print(f"RESULT distinct buildings: {d['distinct_bins']:,.0f}")
d[["window_filed_complaints", "months_in_window", "monthly_average", "daily_average", "distinct_bins"]]

RESULT complaints per month (avg): 10,173
RESULT complaints per day (avg): 334
RESULT distinct buildings: 219,413


metric
window_filed_complaints    783355.0
months_in_window               77.0
monthly_average             10173.4
daily_average                 334.0
distinct_bins              219413.0
Name: value, dtype: float64

## Outcome shares across all complaints

Disposition mix across all complaints and the ECB-linked share (`make_descriptive_figures.py`, `post0_descriptive_stats.py`; database-derived, displayed from committed tables).

In [4]:
sh = committed("desc_outcome_shares.csv")
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
for k in ["violation", "no_violation", "no_access"]:
    print(f"RESULT {k}: {sh.loc[sh.outc==k,'share'].iloc[0]*100:.1f}%")
print(f"RESULT ECB-linked: {d['share_ecb_linked']*100:.1f}%")
sh

RESULT violation: 22.3%
RESULT no_violation: 48.2%
RESULT no_access: 17.7%
RESULT ECB-linked: 21.4%


,outc,n,share
0,no_access,138288,0.176548
1,no_violation,377531,0.481982
2,other,28873,0.036861
3,pending,926,0.001182
4,referral,63267,0.080771
5,violation,174404,0.222656


## ECB penalties imposed

Total ECB/OATH penalties imposed 2020 - May 2026, the share of citations carrying a penalty, and the median penalty (`ecb_penalty_stats.py`).

In [5]:
e = committed("ecb_penalty_stats.csv").iloc[0]
print(f"RESULT penalties imposed: ${e['penalties_imposed_usd']/1e6:,.0f} million")
print(f"RESULT share of citations with a penalty: {e['share_with_positive_penalty']*100:.0f}%")
print(f"RESULT median penalty: ${e['median_positive_penalty_usd']:,.0f}")
committed("ecb_penalty_stats.csv").T

RESULT penalties imposed: $976 million
RESULT share of citations with a penalty: 77%
RESULT median penalty: $2,500


,0
window,20200101-20260531
n_ecb_citations,369453
n_with_positive_penalty,283406
share_with_positive_penalty,0.767096
penalties_imposed_usd,975775969.0
penalties_paid_usd,334873045.0
median_positive_penalty_usd,2500.0
dob_violations_penalty_columns,0


## Illegal conversion is the biggest category

Illegal-conversion complaints (category 45): count, no-access rate, and violation rate (`make_descriptive_figures.py`).

In [6]:
c = committed("desc_category_outcomes.csv").set_index("code")
r = c.loc["45"]
print(f"RESULT illegal-conversion complaints: {r['n']:,.0f}")
print(f"RESULT end with no access: {r['noacc']*100:.0f}%")
print(f"RESULT end in a violation: {r['viol']*100:.1f}%")
c.loc[["45"]]

RESULT illegal-conversion complaints: 77,378
RESULT end with no access: 66%
RESULT end in a violation: 9.7%


,n,viol,noviol,noacc
code,,,,
45,77378,0.097418,0.151025,0.660769


## Elevator complaint outcomes

Elevator complaints (category 6S): no-access and violation rates, the mirror image of illegal conversion (`make_descriptive_figures.py`).

In [7]:
c = committed("desc_category_outcomes.csv").set_index("code")
r = c.loc["6S"]
print(f"RESULT no access: {r['noacc']*100:.1f}%")
print(f"RESULT end in a violation: {r['viol']*100:.0f}%")
c.loc[["6S"]]

RESULT no access: 0.5%
RESULT end in a violation: 35%


,n,viol,noviol,noacc
code,,,,
6S,46453,0.349945,0.360795,0.005489


## Agency initiated inspection outcomes

DOB's own compliance inspections (category 8A) versus broad construction sweeps: violation and no-violation rates (`make_descriptive_figures.py`).

In [8]:
c = committed("desc_category_outcomes.csv").set_index("code")
print(f"RESULT compliance inspections (8A) end in a violation: {c.loc['8A','viol']*100:.0f}%")
sweep = c.sort_values("noviol", ascending=False).iloc[0]
print(f"RESULT broadest-sweep category ({sweep.name}) ends with no violation: {sweep['noviol']*100:.0f}%")
c.sort_values("noviol", ascending=False).head(4)

RESULT compliance inspections (8A) end in a violation: 57%
RESULT broadest-sweep category (7G) ends with no violation: 90%


,n,viol,noviol,noacc
code,,,,
7G,25787,0.088145,0.901695,0.000582
04,18587,0.033141,0.857373,0.056437
23,22308,0.281737,0.666039,0.001390
7J,33193,0.114813,0.574910,0.247040


## After hours construction outcomes

Outcome mix and inspection timing for after-hours construction complaints (category 04): no-violation and violation shares, same-day inspection rate, and median days to inspection (`post0_descriptive_stats.py`).

In [9]:
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
print(f"RESULT no violation found: {d['afterhours_share_noviol']*100:.0f}%")
print(f"RESULT end in a violation: {d['afterhours_share_viol']*100:.1f}%")
print(f"RESULT inspected the same day: {d['afterhours_share_same_day']*100:.0f}%")
print(f"RESULT median days to inspection: {d['afterhours_median_days_to_inspection']:.0f}")

RESULT no violation found: 86%
RESULT end in a violation: 3.3%
RESULT inspected the same day: 11%
RESULT median days to inspection: 4


## Complaints per building by borough

Complaints per residential tax lot by borough; Manhattan leads because a Manhattan lot is often a whole tower (`post0_descriptive_stats.py`).

In [10]:
pl = (panel.groupby("borough").apply(lambda g: g.n_complaints.sum()/len(g))
      .sort_values(ascending=False))
print(pl.round(2).to_string())
print(f"\nRESULT Manhattan per lot: {pl['MN']:.1f}   Queens per lot: {pl['QN']:.1f}"
      f"   ratio: {pl['MN']/pl['QN']:.0f}x")

borough
MN    3.89
BX    1.23
BK    0.91
QN    0.58
SI    0.27

RESULT Manhattan per lot: 3.9   Queens per lot: 0.6   ratio: 7x


## Complaints per unit by borough

Complaints per 100 residential units by borough; the ranking inverts once you divide by apartments (`make_descriptive_figures.py`).

In [11]:
pu = (panel.groupby("borough").apply(lambda g: g.n_complaints.sum()/g.unitsres.sum()*100)
      .sort_values(ascending=False))
print(pu.round(1).to_string())
print(f"\nRESULT Brooklyn {pu['BK']:.1f}  Queens {pu['QN']:.1f}  Manhattan {pu['MN']:.1f} per 100 units")

borough
BK    20.3
QN    19.5
SI    16.6
BX    15.7
MN    13.1

RESULT Brooklyn 20.3  Queens 19.5  Manhattan 13.1 per 100 units


## Complaints and violations per unit by building size

Per-100-unit complaint and disposition-violation rates by building size class (`make_descriptive_figures.py`).

In [12]:
def ubin(u):
    if u == 1: return "1 unit"
    if u == 2: return "2"
    if u <= 4: return "3-4"
    if u <= 10: return "5-10"
    if u <= 20: return "11-20"
    if u <= 50: return "21-50"
    if u <= 100: return "51-100"
    return "100+"
p = panel.copy(); p["ub"] = p.unitsres.map(ubin)
order = ["1 unit", "2", "3-4", "5-10", "11-20", "21-50", "51-100", "100+"]
g = (p.groupby("ub").apply(lambda x: pd.Series({
        "compl_per100": x.n_complaints.sum()/x.unitsres.sum()*100,
        "viol_per100": x.n_viol_disp.sum()/x.unitsres.sum()*100}))
     .reindex(order))
print(g.round(1).to_string())
print(f"\nRESULT single-family: {g.loc['1 unit','compl_per100']:.1f} complaints / "
      f"{g.loc['1 unit','viol_per100']:.1f} violations per 100 units")
print(f"RESULT 100+ units: {g.loc['100+','compl_per100']:.1f} / {g.loc['100+','viol_per100']:.1f}")
print(f"RESULT 5-10 unit walk-up: {g.loc['5-10','compl_per100']:.1f} / {g.loc['5-10','viol_per100']:.1f}")

        compl_per100  viol_per100
ub                               
1 unit          33.9          6.5
2               26.5          4.8
3-4             22.0          4.7
5-10            26.2          6.2
11-20           19.5          5.0
21-50           16.7          4.4
51-100          13.8          3.4
100+             6.7          1.2

RESULT single-family: 33.9 complaints / 6.5 violations per 100 units
RESULT 100+ units: 6.7 / 1.2
RESULT 5-10 unit walk-up: 26.2 / 6.2


## Neighborhood variation across tracts

Complaints per 100 units across census tracts with at least 200 residential units: the 90th vs 10th percentile spread (`post0_descriptive_stats.py`).

In [13]:
tr = (panel.groupby("bct2020").agg(compl=("n_complaints", "sum"), units=("unitsres", "sum")))
tr = tr[tr.units >= 200]; tr["per100"] = tr.compl/tr.units*100
p90, p10 = np.percentile(tr.per100, 90), np.percentile(tr.per100, 10)
print(f"RESULT tracts (>=200 units): {len(tr):,}")
print(f"RESULT 90th percentile: {p90:.0f} complaints per 100 units")
print(f"RESULT 10th percentile: {p10:.1f} complaints per 100 units")
print(f"RESULT ratio: {p90/p10:.1f}x")

RESULT tracts (>=200 units): 2,206
RESULT 90th percentile: 32 complaints per 100 units
RESULT 10th percentile: 8.7 complaints per 100 units
RESULT ratio: 3.7x


## Observed to expected complaints by tract

The residual map (`make_complaint_maps.py`) fits a building-characteristics Poisson with no geographic terms, then maps observed / expected complaints per tract. This cell refits that exact model and lists the tracts and neighborhoods its extremes name: South Ozone Park tract by tract (hottest first), the 12 hottest tracts city-wide, and the quietest neighborhoods. Tract to neighborhood (NTA) names come from a committed crosswalk derived from `data/nyct2020.geojson` (NYC Dept. of City Planning).

In [14]:
import statsmodels.api as sm
# Expectation = the residual map's building-only Poisson (make_complaint_maps.tract_table):
# the 19 building covariates + log2(units) + exact size-bin dummies. Geography enters
# nowhere, so observed / expected shows where volume runs above what the stock predicts.
d = frame.copy()
size_d = pd.get_dummies(d["size_bin"], prefix="sz", drop_first=True).astype(float)
d["log2_units"] = np.log2(d["unitsres"].clip(lower=1))
Xd = sm.add_constant(pd.concat([d[BUILDING].astype(float), d[["log2_units"]], size_d], axis=1))
glm = sm.GLM(d["n_complaints"].astype(float), Xd, family=sm.families.Poisson()).fit()
d["expected"] = glm.predict(Xd)
print(f"Poisson fit on {len(d):,} lots; observed {d.n_complaints.sum():,.0f} "
      f"= expected {d.expected.sum():,.0f} by construction\n")

# Collapse to 2020 census tracts, attach NTA neighborhood names, keep >= 200 resid. units
xw = committed("tract_nta_crosswalk.csv"); xw["boroct2020"] = xw["boroct2020"].astype(str)
t = (d.groupby("bct2020").agg(obs=("n_complaints", "sum"), exp=("expected", "sum"),
                              units=("unitsres", "sum")).reset_index()
       .merge(xw, left_on="bct2020", right_on="boroct2020", how="left"))
t = t[t.units >= 200].copy(); t["ratio"] = t.obs / t.exp
FMT = {"exp": "{:.0f}".format, "ratio": "{:.1f}x".format}

# 1. South Ozone Park -- the illegal-conversion center -- tract by tract, hottest first
sop = t[t.ntaname.str.contains("Ozone Park", case=False, na=False)].sort_values("ratio", ascending=False)
print("South Ozone Park / Ozone Park tracts -- observed / expected complaints, hottest first:")
print(sop[["ntaname", "bct2020", "obs", "exp", "ratio"]].head(15).to_string(index=False, formatters=FMT))

# 2. The 12 hottest tracts city-wide: South Ozone Park on top, then central Bronx and Brooklyn
print("\n12 hottest tracts city-wide (observed / expected):")
print(t.nlargest(12, "ratio")[["boroname", "ntaname", "obs", "exp", "ratio"]].to_string(index=False, formatters=FMT))

# 3. The quietest NEIGHBORHOODS (NTA aggregates): Staten Island, SE Queens, Upper East/West Sides
nta = (t.groupby(["boroname", "ntaname"]).agg(obs=("obs", "sum"), exp=("exp", "sum"))
        .assign(ratio=lambda x: x.obs / x.exp).reset_index())
nta = nta[nta.obs >= 100]
print("\n10 quietest neighborhoods (NTA observed / expected):")
print(nta.nsmallest(10, "ratio")[["boroname", "ntaname", "obs", "exp", "ratio"]]
      .to_string(index=False, formatters={"exp": "{:.0f}".format, "ratio": "{:.2f}x".format}))

r = sop.ratio.tolist()
lau = float(nta.loc[nta.ntaname == "Laurelton", "ratio"].iloc[0])
ros = float(nta.loc[nta.ntaname == "Rosedale", "ratio"].iloc[0])
print(f"\nRESULT South Ozone Park's hottest tracts file several times their predicted complaint "
      f"volume (peak tract {r[0]:.1f}x, then {r[1]:.1f}x, {r[2]:.1f}x, {r[3]:.1f}x); the quiet "
      f"tract-house neighborhoods above -- Staten Island, and southeast Queens like Laurelton "
      f"({lau:.2f}) and Rosedale ({ros:.2f}) -- file about one half to two thirds of predicted")

Poisson fit on 766,429 lots; observed 647,506 = expected 647,506 by construction

South Ozone Park / Ozone Park tracts -- observed / expected complaints, hottest first:
         ntaname bct2020  obs exp ratio
South Ozone Park 4010000 2349 297  7.9x
South Ozone Park 4016600 1155 290  4.0x
South Ozone Park 4017000  681 202  3.4x
South Ozone Park 4009600  868 284  3.1x
South Ozone Park 4016400  733 266  2.8x
South Ozone Park 4083800 1139 420  2.7x
South Ozone Park 4015802 1059 401  2.6x
South Ozone Park 4010400  758 290  2.6x
South Ozone Park 4084602  158  63  2.5x
South Ozone Park 4017800  407 167  2.4x
South Ozone Park 4010600  554 281  2.0x
South Ozone Park 4009400  506 265  1.9x
South Ozone Park 4010200  429 229  1.9x
South Ozone Park 4015801  592 321  1.8x
South Ozone Park 4009800  469 261  1.8x

12 hottest tracts city-wide (observed / expected):
boroname                                   ntaname  obs exp ratio
  Queens                          South Ozone Park 2349 297  7.9x
   Bron

## Complaint and inspector text coverage

The scrape universe and coverage -- complaints filed in the window, pages actually scraped, and the count and share carrying an inspection report -- then the caller's complaint-text coverage and the distinct inspector count (`post0_descriptive_stats.py`). The counts here are the TLDR's 783,355 / 783,289 / 778,312.

In [15]:
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
filed, scraped = d["window_filed_complaints"], d["scraped_pages"]
report, subj = d["n_with_inspection_comments"], d["n_with_complaint_text"]
print(f"RESULT complaints filed Jan 2020-May 2026: {filed:,.0f}")
print(f"RESULT scraped complaint pages: {scraped:,.0f}  ({scraped/filed*100:.2f}% of filed)")
print(f"RESULT carry an inspection report (inspector narrative): {report:,.0f}  ({report/scraped*100:.1f}%)")
print(f"RESULT carry the caller's complaint text: {subj:,.0f}  ({subj/scraped*100:.1f}%)")
print(f"RESULT distinct inspector badges: {d['distinct_inspector_badges']:,.0f}")
# the committed rounded shares must match the counts recomputed here
assert abs(d["scrape_coverage_share"] - scraped/filed) < 1e-6
assert abs(d["share_with_inspection_comments"] - round(report/scraped, 4)) < 1e-6
assert abs(d["share_with_complaint_text"] - round(subj/scraped, 4)) < 1e-6

RESULT complaints filed Jan 2020-May 2026: 783,355
RESULT scraped complaint pages: 783,289  (99.99% of filed)
RESULT carry an inspection report (inspector narrative): 778,312  (99.4%)
RESULT carry the caller's complaint text: 616,614  (78.7%)
RESULT distinct inspector badges: 815


## Where complaints originate

Share of complaints carrying a 311 reference number versus originating inside the agency (`post0_descriptive_stats.py`).

In [16]:
d = committed("post0_descriptive_stats.csv").set_index("metric")["value"].astype(float)
print(f"RESULT 311-referenced: {d['origin_share_311']*100:.0f}%")
print(f"RESULT agency-initiated: {d['origin_share_agency']*100:.0f}%")

RESULT 311-referenced: 68%
RESULT agency-initiated: 32%


## Correlation among the three violation measures

Pairwise correlations among disposition violations, ECB citations, and the deduplicated DOB-violation union (`panel_headline_counts.py`). The first pair recomputes from the panel; the union measure is built from the database, so its correlations are displayed from the committed table.

In [17]:
print(f"RESULT disposition ~ ECB (recomputed): {panel['n_viol_disp'].corr(panel['n_ecb_2020on']):.2f}")
phc = committed("panel_headline_counts.csv").set_index("metric")["value"]
print(f"RESULT disposition ~ DOB union: {phc['pearson_n_viol_disp_vs_n_dobviol_union']:.2f}")
print(f"RESULT ECB ~ DOB union: {phc['pearson_n_ecb_2020on_vs_n_dobviol_union']:.2f}")

RESULT disposition ~ ECB (recomputed): 0.79
RESULT disposition ~ DOB union: 0.42
RESULT ECB ~ DOB union: 0.41


## Violation type composition

Of complaints that end in a violation, the BIS disposition code records which enforcement instrument was issued (`post0_violation_type_composition.py`). Most are OATH/ECB penalty summonses (code A8); a smaller share are DOB Buildings corrective violations (codes A1, A6); a few are both (A9); the rest are other enforcement such as stop-work and vacate orders. This composition is why the disposition-violation measure tracks the ECB measure far more closely than the deduplicated DOB-violation union.

In [18]:
vt = committed("violation_type_composition.csv").set_index("category")
tot = int(vt.loc["total", "n"])
for cat in ["oath_ecb", "buildings_dob", "both", "other_enforcement"]:
    r = vt.loc[cat]
    print(f"RESULT {r['label']}: {int(r['n']):,} ({r['share']*100:.1f}% of {tot:,})")
# the four buckets must partition the total exactly
parts = vt.loc[["oath_ecb", "buildings_dob", "both", "other_enforcement"]]
assert int(parts["n"].sum()) == tot
assert abs(parts["share"].sum() - 1) < 0.001

RESULT OATH / ECB violation (penalty summons): 137,609 (78.9% of 174,404)
RESULT DOB Buildings violation (corrective notice): 12,818 (7.3% of 174,404)
RESULT Both an OATH and a DOB violation: 769 (0.4% of 174,404)
RESULT Other enforcement (stop-work, vacate, criminal summons, deficiency): 23,208 (13.3% of 174,404)


## Raw citation rates by owner type

Raw ECB citations per 100 lots per year by owner type, before any adjustment (`violation_rate_models.py`).

In [19]:
def rate(mask): return frame[mask].n_ecb_2020on.mean()/YEARS*100
print(f"RESULT individually owned: {rate(frame.owner_type=='individual'):.1f} ECB per 100 lots/yr")
print(f"RESULT LLC-owned: {rate(frame.llc==1):.1f} ECB per 100 lots/yr")
print(f"RESULT NYCHA: {rate(frame.nycha==1):.1f} ECB per 100 lots/yr")

RESULT individually owned: 2.6 ECB per 100 lots/yr


RESULT LLC-owned: 20.8 ECB per 100 lots/yr
RESULT NYCHA: 58.1 ECB per 100 lots/yr


## Raw rates by owner occupancy

Raw complaint and per-inspection violation rates for owner-occupied (STAR-enrolled) versus absentee properties, pooled across all sizes (`post0_descriptive_stats.py`).

In [20]:
for f, lab in [(1, "owner-occupied"), (0, "absentee")]:
    g = panel[panel.owner_occ_star == f]
    print(f"RESULT {lab}: {g.n_complaints.sum()/g.unitsres.sum()*100:.1f} complaints per 100 units | "
          f"{g.n_viol_disp.sum()/g.n_substantive.sum()*100:.1f} violations per 100 inspections")

RESULT owner-occupied: 18.2 complaints per 100 units | 27.8 violations per 100 inspections


RESULT absentee: 17.1 complaints per 100 units | 32.4 violations per 100 inspections


## Owner occupancy gap within small homes

Within two-, three-, and four-unit homes, absentee complaint and violation rates per 100 units against owner-occupied homes of the same size (`make_descriptive_figures.py`).

In [21]:
rows = []
for n in [2, 3, 4]:
    s = panel[panel.unitsres == n]
    for f, lab in [(1, "owner-occ"), (0, "absentee")]:
        g = s[s.owner_occ_star == f]
        rows.append(dict(units=n, group=lab,
                         compl_per100=g.n_complaints.sum()/g.unitsres.sum()*100,
                         viol_per100=g.n_viol_disp.sum()/g.unitsres.sum()*100))
tab = pd.DataFrame(rows); print(tab.round(1).to_string(index=False))
piv = tab.pivot(index="units", columns="group")
cl = (piv[("compl_per100", "absentee")]/piv[("compl_per100", "owner-occ")]-1)*100
vl = (piv[("viol_per100", "absentee")]/piv[("viol_per100", "owner-occ")]-1)*100
print(f"\nRESULT absentee complaint excess per unit: {cl.min():.0f}-{cl.max():.0f}% across 2-4 unit homes")
print(f"RESULT absentee violation excess per unit: {vl.min():.0f}-{vl.max():.0f}% across 2-4 unit homes")

 units     group  compl_per100  viol_per100
     2 owner-occ          23.0          3.9
     2  absentee          28.8          5.4
     3 owner-occ          17.3          3.3
     3  absentee          24.1          5.2
     4 owner-occ          20.5          4.1
     4  absentee          25.6          5.9

RESULT absentee complaint excess per unit: 25-39% across 2-4 unit homes
RESULT absentee violation excess per unit: 37-56% across 2-4 unit homes


## LLC complaint premium

PPML complaint count, LLC vs individually owned, within census tract and exact unit-count size class, commercial exposure held constant (`risk_factor_models.py`, model `tract_ppml_ncomp`). Refit here on the committed panel.

In [22]:
m = pf.fepois(f"n_complaints ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "llc")
print(f"RESULT LLC complaints: {v:+.0f}%   95% CI [{lo:.0f}, {hi:.0f}]   N={m._N:,}")
m.tidy().loc[["llc"]]

RESULT LLC complaints: +70%   95% CI [64, 76]   N=766,382


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
llc,0.530268,0.019191,27.631478,0.0,0.492655,0.567882


## Owner occupied complaint gap in the full model

Same PPML complaint model, owner-occupied (STAR) coefficient: the complaint gap is close to zero once size and commercial use are held constant (`risk_factor_models.py`).

In [23]:
m = pf.fepois(f"n_complaints ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "owner_occ_star")
print(f"RESULT owner-occupied complaint gap: {v:+.1f}%   95% CI [{lo:+.0f}, {hi:+.0f}]   N={m._N:,}")
m.tidy().loc[["owner_occ_star"]]

RESULT owner-occupied complaint gap: +0.1%   95% CI [-3, +4]   N=766,382


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
owner_occ_star,0.000571,0.017634,0.032398,0.974154,-0.03399,0.035132


## Owner occupied inspection citation gap

Weighted, category-adjusted LPM of the per-inspection violation rate; owner-occupied coefficient in percentage points (`risk_factor_models.py`, model `cond_violrate_catadj`).

In [24]:
sub = frame[frame.n_substantive > 0]
m = pf.feols(f"violrate100 ~ {XC} | {FE}", data=sub, weights="n_substantive", vcov=VCOV)
b, lo, hi = pp(m, "owner_occ_star")
print(f"RESULT owner-occupied inspection gap: {b:+.1f} pp   95% CI [{lo:.1f}, {hi:.1f}]   N={m._N:,}")
m.tidy().loc[["owner_occ_star"]]

RESULT owner-occupied inspection gap: -2.8 pp   95% CI [-3.4, -2.2]   N=135,576


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
owner_occ_star,-2.798262,0.324057,-8.6351,0.0,-3.433753,-2.162771


## LLC ECB citation premium

PPML ECB citations, LLC vs individually owned, same fixed effects (`violation_rate_models.py`, model `ppml_ecb`). Refit on the committed panel.

In [25]:
m = pf.fepois(f"n_ecb_2020on ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "llc")
print(f"RESULT LLC ECB citations: {v:+.0f}%   95% CI [{lo:.0f}, {hi:.0f}]   N={m._N:,}")
m.tidy().loc[["llc"]]

RESULT LLC ECB citations: +103%   95% CI [94, 111]   N=766,376


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
llc,0.705804,0.021305,33.128557,0.0,0.664047,0.747561


## Prior violations predict future citations

Same ECB model, prior-violation (2010-2019) coefficient: past enforcement is the strongest single predictor of future citations (`violation_rate_models.py`).

In [26]:
m = pf.fepois(f"n_ecb_2020on ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "any_prior_viol")
print(f"RESULT prior-violation ECB premium: {v:+.0f}%   95% CI [{lo:.0f}, {hi:.0f}]   N={m._N:,}")
m.tidy().loc[["any_prior_viol"]]

RESULT prior-violation ECB premium: +121%   95% CI [112, 130]   N=766,376


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
any_prior_viol,0.791729,0.021519,36.792233,0.0,0.749553,0.833905


## Trust and estate held properties

Same ECB model, trust/estate coefficient: fewer citations than individually owned buildings (`violation_rate_models.py`).

In [27]:
m = pf.fepois(f"n_ecb_2020on ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "trust_estate")
print(f"RESULT trust/estate ECB gap: {v:+.0f}%  (i.e. {-v:.0f}% fewer)   95% CI [{lo:.0f}, {hi:.0f}]   N={m._N:,}")
m.tidy().loc[["trust_estate"]]

RESULT trust/estate ECB gap: -27%  (i.e. 27% fewer)   95% CI [-34, -18]   N=766,376


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
trust_estate,-0.308012,0.05329,-5.779907,7.474198e-09,-0.412458,-0.203565


## NYCHA developments

Same ECB model, NYCHA coefficient: the 22-fold raw gap reverses once building size enters, leaving fewer citations than comparable individually owned buildings (`violation_rate_models.py`).

In [28]:
m = pf.fepois(f"n_ecb_2020on ~ {X} | {FE}", data=frame, vcov=VCOV)
v, lo, hi = irr(m, "nycha")
print(f"RESULT NYCHA ECB gap: {v:+.0f}%  (i.e. {-v:.0f}% fewer)   95% CI [{lo:.0f}, {hi:.0f}]   N={m._N:,}")
m.tidy().loc[["nycha"]]

RESULT NYCHA ECB gap: -21%  (i.e. 21% fewer)   95% CI [-31, -8]   N=766,376


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
nycha,-0.230614,0.073884,-3.121319,0.0018,-0.375423,-0.085805


## Distant owners

Owner-augmented ECB model: owners whose deed address is elsewhere in NYC, or outside NYC, versus same-ZIP owners (`violation_rate_models.py`, `ppml_ecb_owner`).

In [29]:
m = pf.fepois(f"n_ecb_2020on ~ {XO} | {FE}", data=frame, vcov=VCOV)
for term, lab in [("geo_nyc_other", "elsewhere in NYC"), ("geo_outside_nyc", "outside NYC")]:
    v, lo, hi = irr(m, term)
    print(f"RESULT owner {lab}: {v:+.0f}% ECB   95% CI [{lo:.0f}, {hi:.0f}]")
print(f"N={m._N:,}")
m.tidy().loc[["geo_nyc_other", "geo_outside_nyc"]]

RESULT owner elsewhere in NYC: +9% ECB   95% CI [5, 13]
RESULT owner outside NYC: +12% ECB   95% CI [7, 17]
N=766,376


,Estimate,Std. Error,t value,Pr(>|t|),2.5%,97.5%
Coefficient,,,,,,
geo_nyc_other,0.086406,0.017238,5.012508,5.372516e-07,0.052620,0.120192
geo_outside_nyc,0.112260,0.024379,4.604782,4.128981e-06,0.064478,0.160042


## Owner occupied DOB violation gap

PPML on the deduplicated DOB-violation union, owner-occupied coefficient (`violation_rate_models.py`, model `ppml_dobviol`). The union outcome is built from the database, so this value is displayed from the committed estimate table; it regenerates by running that script against the database snapshot.

In [30]:
c = committed("citation_tidy_estimates.csv")
r = c[(c.model == "ppml_dobviol") & (c.term == "owner_occ_star")].iloc[0]
v, lo, hi = r["pct_change"], (np.exp(r["25pct"])-1)*100, (np.exp(r["975pct"])-1)*100
print(f"RESULT owner-occupied DOB-violation gap: {v:+.1f}%  (i.e. {-v:.0f}% fewer)   "
      f"95% CI [{lo:.0f}, {hi:.0f}]   N={r['n']:,.0f}")
c[(c.model == 'ppml_dobviol') & (c.term == 'owner_occ_star')]

RESULT owner-occupied DOB-violation gap: -10.4%  (i.e. 10% fewer)   95% CI [-13, -7]   N=765,821


,term,estimate,std_error,t_value,pr(>|t|),25pct,975pct,model,outcome,n,pct_change
24,owner_occ_star,-0.110072,0.016588,-6.635582,3.232259e-11,-0.142584,-0.07756,ppml_dobviol,DOB violations 2020-26 (union),765821,-10.423001


## Owner occupancy gap without commercial controls

The owner-occupied DOB-violation gap WITHOUT the commercial-exposure controls, for contrast (`owner_commercial_sensitivity.py`, spec 1). Built from the database; displayed from the committed sensitivity table.

In [31]:
c = pd.read_csv(DATA / "owner_commercial_sensitivity.csv")
r = c[(c.okey == "dobviol") & (c.owner == "owner_occ_star")].iloc[0]
b, se = float(r.s1_b), float(r.s1_se)
v = (np.exp(b)-1)*100; lo = (np.exp(b-1.96*se)-1)*100; hi = (np.exp(b+1.96*se)-1)*100
print(f"RESULT owner-occupied gap, no commercial controls: {v:+.0f}%  (i.e. {-v:.0f}% fewer)   "
      f"95% CI [{lo:.0f}, {hi:.0f}]")
c[(c.okey == 'dobviol') & (c.owner == 'owner_occ_star')][['outcome', 'okey', 'owner', 's1_b', 's1_se', 's1_eff']]

RESULT owner-occupied gap, no commercial controls: -22%  (i.e. 22% fewer)   95% CI [-25, -19]


,outcome,okey,owner,s1_b,s1_se,s1_eff
7,DOB-ledger violation count,dobviol,owner_occ_star,-0.24708,0.017528,-21.892153


## Verification against committed estimates

Each refit above is cross-checked against the committed tidy-estimate tables and the article's quoted number. All rows should read PASS.

In [32]:
te, ce = committed("tidy_estimates.csv"), committed("citation_tidy_estimates.csv")
def logpct(df, model, term):
    r = df[(df.model == model) & (df.term == term)].iloc[0]
    return (np.exp(r.estimate)-1)*100
def ppval(df, model, term):
    return df[(df.model == model) & (df.term == term)].iloc[0].estimate
checks = [
    ("LLC complaints",        logpct(te, "tract_ppml_ncomp", "llc"),            70),
    ("Owner-occ complaint",   logpct(te, "tract_ppml_ncomp", "owner_occ_star"),  0),
    ("Owner-occ inspection",  ppval(te, "cond_violrate_catadj", "owner_occ_star"), -2.8),
    ("LLC ECB",               logpct(ce, "ppml_ecb", "llc"),                    103),
    ("Prior-viol ECB",        logpct(ce, "ppml_ecb", "any_prior_viol"),         121),
    ("Trust/estate ECB",      logpct(ce, "ppml_ecb", "trust_estate"),           -27),
    ("NYCHA ECB",             logpct(ce, "ppml_ecb", "nycha"),                  -21),
    ("Owner elsewhere NYC",   logpct(ce, "ppml_ecb_owner", "geo_nyc_other"),      9),
    ("Owner outside NYC",     logpct(ce, "ppml_ecb_owner", "geo_outside_nyc"),   12),
    ("Owner-occ DOB",         logpct(ce, "ppml_dobviol", "owner_occ_star"),     -10),
]
out = pd.DataFrame([{"result": n, "committed": round(c, 1), "article": a,
                     "check": "PASS" if abs(round(c)-a) <= 1 else "REVIEW"}
                    for n, c, a in checks])
print(out.to_string(index=False))

              result  committed  article check
      LLC complaints       69.9     70.0  PASS
 Owner-occ complaint        0.1      0.0  PASS
Owner-occ inspection       -2.8     -2.8  PASS
             LLC ECB      102.5    103.0  PASS
      Prior-viol ECB      120.7    121.0  PASS
    Trust/estate ECB      -26.5    -27.0  PASS
           NYCHA ECB      -20.6    -21.0  PASS
 Owner elsewhere NYC        9.0      9.0  PASS
   Owner outside NYC       11.9     12.0  PASS
       Owner-occ DOB      -10.4    -10.0  PASS
